# Day 1 Exercise - Guided Tasks

Build a data pipeline for Singapore taxi availability and 2-hour weather forecasts. The final output should be a DuckDB database that the dashboard can read.


## Target Workflow

1. Collect taxi data from an API.
2. Collect weather data from an API.
3. Save raw API responses to files.
4. Clean taxi and weather data.
5. Transform taxi coordinates into planning-area counts.
6. Save the two structured business tables into DuckDB.
7. Query DuckDB with `SELECT` to verify the data.
8. Join taxi and weather tables into one analytical view.
9. Verify that the dashboard can read the database.


## Data Sources

Use the Singapore data.gov.sg datasets below as your starting point.

- 2-hour Weather Forecast: https://data.gov.sg/datasets?query=weather&resultId=d_3f9e064e25005b0e42969944ccaf2e7a
- Taxi Availability: https://data.gov.sg/datasets?query=taxi&resultId=d_e25662f1a062dd046453926aa284ba64

Open each dataset page, inspect the available API endpoint, and decide what URL your notebook should call.


## Task 1 - Setup

Create imports, constants, and folders.

Hints:

- Use paths relative to this notebook.
- Create `data/raw` and `data/processed`.
- Install and import DuckDB if needed: `%pip install duckdb`.
- The database should be named `day_1_weather_taxi_data.duckdb`.
- Read an API key from `DATA_GOV_API_KEY` or `shared_assets/api_key.txt` only if needed.


## Task 2 - Fetch Taxi Availability

Call the taxi availability endpoint and inspect the JSON structure.

Hints:

- Check the HTTP status before processing.
- Taxi coordinates are nested inside the first feature.
- Save the raw JSON data to a CSV file before cleaning it so you can inspect the original API output.


## Task 3 - Fetch Weather Forecast

Call the 2-hour weather forecast endpoint and inspect the JSON structure.

Hints:

- Weather forecast rows are inside the first item.
- Each forecast has an `area` and `forecast`.
- Save the raw JSON data to a CSV file before cleaning it so you can inspect the original API output.


## Before Task 4 - Read the Taxi API Schema

Before cleaning the taxi data, inspect the schema so you know where the useful fields live.

Important parts of the Taxi Availability schema:

```json
{
  "type": "object",
  "description": "A GeoJSON representing the locations of available taxis in Singapore",
  "properties": {
    "features": {
      "type": "array",
      "items": {
        "properties": {
          "geometry": {
            "properties": {
              "type": {"enum": ["MultiPoint"]},
              "coordinates": {
                "type": "array",
                "items": {
                  "type": "array",
                  "description": "A position (longitude, latitude)",
                  "minItems": 2,
                  "maxItems": 2
                }
              }
            }
          },
          "properties": {
            "properties": {
              "timestamp": {"type": "string", "format": "date-time"},
              "taxi_count": {"type": "integer"}
            }
          }
        }
      }
    }
  }
}
```

Cleaning target:

- `features[0].geometry.coordinates` becomes many taxi-location rows.
- Each coordinate pair is ordered as `[longitude, latitude]`.
- `features[0].properties.timestamp` should be kept as the API snapshot time.
- `features[0].properties.taxi_count` can be used as a sanity check against the number of coordinate rows.


## Task 4 - Clean Taxi Data

Convert the taxi coordinate list into a DataFrame.

Hints:

- Use columns named `longitude` and `latitude`.
- Convert both columns to numeric values.
- Remove invalid or missing coordinates.
- Keep the original taxi API timestamp.
- Save the cleaned points to `data/processed`.


## Before Task 5 - Read the Weather API Structure

Before cleaning the weather data, inspect the JSON structure and decide what one row should represent.

Important parts of the 2-hour Weather Forecast response:

```json
{
  "items": [
    {
      "update_timestamp": "...",
      "valid_period": {
        "start": "...",
        "end": "..."
      },
      "forecasts": [
        {"area": "Ang Mo Kio", "forecast": "Cloudy"},
        {"area": "Bedok", "forecast": "Fair"}
      ]
    }
  ]
}
```

Cleaning target:

- `items[0].forecasts` becomes many weather rows.
- Each row should represent one planning area forecast.
- Repeat `update_timestamp`, `valid_period.start`, and `valid_period.end` on every row.
- Keep area names in a readable format, for example `Ang Mo Kio`.


## Task 5 - Clean Weather Data

Flatten the weather API response into one row per area.

Hints:

- Include update timestamp, valid period start, valid period end, area, and forecast.
- Convert timestamps into a consistent ISO format.
- Strip extra whitespace from text columns.
- Save the cleaned weather data to `data/processed`.


## Task 6 - Spatial Transformation

At the end of Task 4, you should have a cleaned taxi-point table that looks roughly like this:

| longitude | latitude |
|---:|---:|
| 103.839 | 1.375 |
| 103.902 | 1.321 |
| 103.765 | 1.438 |

Each row is one available taxi location. The problem is that longitude and latitude are coordinates, but our dashboard wants a more human-friendly summary such as:

| planning_area | available_taxi_count |
|---|---:|
| Ang Mo Kio | 12 |
| Bedok | 34 |
| Jurong West | 21 |

So the question is: how do we convert a taxi point into a Singapore planning area?

A taxi point alone is not enough. We also need boundary data that tells us which polygon belongs to which planning area. In other words, we need a map layer where each area has a shape made from many coordinate pairs.

Data Source 2: Singapore Boundary Data  
https://data.gov.sg/datasets/d_8594ae9ff96d0c708bc2af633048edfb/view

Download the boundary data and place the GeoJSON file in this exercise folder. In this notebook, we will use:

`shared_assets/MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson`

Before writing code, open the file and inspect its structure. A GeoJSON file is still JSON, but it follows a geospatial convention:

- A `FeatureCollection` contains many `Feature` objects.
- Each `Feature` has `properties`, such as the planning area name.
- Each `Feature` has `geometry`, such as a Polygon or MultiPolygon boundary.
- A taxi location is a Point.
- Spatial transformation asks: which boundary polygon contains this point?

In Python, a common library for this is GeoPandas. It extends pandas with geometry columns and spatial operations. The key operation here is a spatial join: match each taxi point to the planning-area polygon that contains it.

A spatial join is similar to a normal table join, but the matching condition is geometric instead of textual. A normal join might ask whether `area_name == planning_area`. A spatial join asks a location question, such as: does this taxi point fall inside this planning-area polygon? After the join, each taxi row can carry the planning area name from the boundary file.

Hints:

- Load the boundary GeoJSON with GeoPandas.
- Convert the taxi DataFrame into a GeoDataFrame of point geometries.
- Make sure taxi points and boundaries use the same coordinate reference system, usually `EPSG:4326` for longitude and latitude.
- Use a spatial join to attach the planning area name to each taxi point.
- Group by planning area and count taxis.
- Save the taxi counts to `data/processed`.


### Task 6 Check - Plot a Simple Map

After your spatial join, make a quick visual check.

Draw the Singapore planning-area boundaries in a light color, then draw taxi points as red dots. The goal is not to make a beautiful production map. The goal is to check whether the data makes sense spatially.

What to look for:

- Taxi points should sit inside or near Singapore boundaries.
- Dense transport areas should have visibly more red points.
- Changi Airport often has many taxi points, so a red cluster in the east is a useful sanity check.
- If points appear in the ocean or far away from Singapore, check longitude/latitude order and CRS.


## Task 7 - Create DuckDB Tables

Before we create a database, look at the two structured CSV files produced by the cleaning and transformation steps.

These CSV files are already regular tables. The database tables should match them closely.

### Table 1: Taxi availability by area

Source CSV: `data/processed/taxi_counts_by_planning_area_latest.csv`

One row means: taxi availability for one planning area at one API snapshot time.

| Column | Meaning |
|---|---|
| `api_timestamp` | Time of the taxi API snapshot |
| `planning_area` | Singapore planning area, such as `Changi` or `Ang Mo Kio` |
| `available_taxi_count` | Number of available taxis in that planning area |

### Table 2: Weather forecast by area

Source CSV: `data/processed/weather_forecasts_latest.csv`

One row means: weather forecast for one area during one forecast validity period.

| Column | Meaning |
|---|---|
| `api_update_timestamp` | Time when the weather API data was updated |
| `valid_period_start` | Start time of the forecast period |
| `valid_period_end` | End time of the forecast period |
| `area` | Singapore area, such as `Changi` or `Ang Mo Kio` |
| `forecast` | Weather forecast text |

Now create dashboard-ready DuckDB tables for these structured outputs.

Required dashboard tables:

- `weather_forecasts`
- `taxi_availability`
- `collection_runs`

`collection_runs` is a pipeline log table. It records when the data collection ran, which source was collected, whether the run succeeded, and how many rows were inserted.

Hints:

- Add uniqueness rules so reruns do not duplicate the same API snapshot.
- DuckDB supports `UNIQUE(...)` constraints and `ON CONFLICT DO NOTHING` for duplicate-safe inserts.
- The dashboard expects fields such as `api_update_timestamp`, `api_timestamp`, `area`, `planning_area`, and `available_taxi_count`.


## Task 8 - Insert Processed Data

Insert the cleaned and transformed data into DuckDB.

Hints:

- Use parameterized SQL.
- Use `ON CONFLICT DO NOTHING` for repeated snapshots.
- Insert one collection log row for weather and one for taxi.


## Task 9 - Select From Your Database

Query the database to confirm it is ready for the dashboard.

Minimum checks:

- Row counts for each table.
- Latest taxi timestamp.
- Latest weather update timestamp.
- Top planning areas by taxi count.


## Task 10 - Join Weather and Taxi Tables

After you can create tables, insert rows, and query them with `SELECT`, move to the next question: how do we combine the taxi table and weather table?

Create an analytical view that combines weather forecast and taxi counts.

Hints:

- Weather areas and planning areas are not always perfect matches.
- Start with a simple uppercase text key.
- Use a left join from weather to taxi counts.
- Fill missing taxi counts with zero.
- Save the merged result to `data/processed`.
- Optionally create a new DuckDB table named `weather_taxi_merged` for the joined output.


## Task 11 - From One-Off Notebook to Scheduled Pipeline

Question: if this notebook runs successfully once, is the dashboard solved permanently?

Probably not. Taxi availability and weather forecasts are live data sources. A dashboard based on one API snapshot will quickly become stale.

Think about what has to happen next:

- The collection logic needs to run repeatedly, for example every few minutes.
- Each run should be idempotent: rerunning should update or skip the same snapshot, not create duplicate rows.
- The job should log when it ran, what source it collected from, and whether it succeeded.
- Failures should be visible so someone can retry or debug the pipeline.

A simple first step is to move the working notebook logic into a Python script and schedule it with a local scheduler such as Windows Task Scheduler or cron. That is enough to automate one task, but it becomes hard to manage when there are many dependent steps.

This leads into Day 2: workflow orchestration. Tools such as Apache Airflow let us define data pipelines as scheduled workflows with tasks, dependencies, retries, logs, and monitoring.

## Task 12 - Run the Dashboard

After your database is ready, run the dashboard from a local terminal. From the `day_1` folder:

```powershell
cd .\04_serving_dashboard
python -m uvicorn serving_api:app --reload --host 127.0.0.1 --port 8000
```

Open `http://localhost:8000` and check whether the dashboard shows taxi and weather data.

Note: this final dashboard step is designed for a local computer, not Colab, because it starts a local web server.
